# 04-01 — Section-aware chunking (real DI + Vision → chunks → embeddings)

This notebook plugs into the same flow as `02_doc_intelligence.ipynb`
and `03_openai_vision.ipynb`, but stops one step further down the
pipeline:

    Blob → Document Intelligence → PyMuPDF + GPT-4o vision →
    **section-aware chunking (with parent-child links)** → **embedding**

It does **not** index into AI Search — that is `04_ai_search.ipynb`.

What's demonstrated here:
- DI now returns `paragraphs`, `sections`, `tables` (with caption + neighbors), and `figures_meta`
- Paragraphs are packed into ~600-token windows that **never cross section boundaries**
- Tables and images bundle their caption + 1–2 nearby paragraphs into the chunk body — so they're never isolated
- Every table / image chunk gets `parent_id` linking it to the section's anchor text chunk
- Multi-PDF identity (`doc_id`, `doc_filename`, `doc_hash`) and layout (`bbox`, `reading_order`) on every chunk

Place a PDF at `notebooks/Multi_Agent_Research_System_Architecture.pdf` (or update `PDF_PATH` below) before running.

## 1. Setup — services + upload PDF


In [1]:
import hashlib
import os
import pathlib
import sys

sys.path.insert(0, os.path.abspath('..'))

from src.blob_client import BlobService
from src.chunking import (
    CHUNK_OVERLAP,
    CHUNK_TOKENS,
    assemble_chunks,
    build_image_chunks,
    build_table_chunks,
    chunk_paragraphs_by_section,
)
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService

PDF_PATH = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF_PATH.exists(), 'Drop a sample PDF next to this notebook'

DOC_ID = 'docmind-chunk-test'
DOC_FILENAME = PDF_PATH.name
BLOB_NAME = f'{DOC_ID}/{DOC_FILENAME}'

blob = BlobService()
blob.ensure_container()
pdf_bytes = PDF_PATH.read_bytes()
DOC_HASH = hashlib.sha256(pdf_bytes).hexdigest()
blob.upload(BLOB_NAME, pdf_bytes, content_type='application/pdf')
url = blob.get_url(BLOB_NAME)

print('PDF URL      :', url[:80], '…')
print('doc_filename :', DOC_FILENAME)
print('doc_hash     :', DOC_HASH[:16], '…')
print(f'CHUNK_TOKENS={CHUNK_TOKENS} (~{CHUNK_TOKENS * 4} chars), CHUNK_OVERLAP={CHUNK_OVERLAP} (~{CHUNK_OVERLAP * 4} chars, ~{CHUNK_OVERLAP * 100 / CHUNK_TOKENS:.0f}%)')


INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '6eaee1a4-4ad5-11f1-88da-bcd22c162bea'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'a1c7a9bf-801e-0094-29e2-dee294000000'
    'x-ms-client-request-id': '6eaee1a4-4ad5-11f1-88da-bcd22c162bea'
    'x-ms-version': 'REDAC

PDF URL      : https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/Mu …
doc_filename : Multi_Agent_Research_System_Architecture.pdf
doc_hash     : 0b4ae90fa7bd2cff …
CHUNK_TOKENS=600 (~2400 chars), CHUNK_OVERLAP=80 (~320 chars, ~13%)


## 2. Document Intelligence — paragraphs, sections, tables, figures

`extract_pdf` now returns the rich layout structure: paragraphs (with bbox + reading order + section_id),
sections (with hierarchy), tables (with caption + neighbor paragraph ids), figures, and `figures_meta`
(section + neighbor lookup keyed by `f<idx>`).


In [2]:
doc_intel = DocIntelService()
result = doc_intel.extract_pdf(url)

text_pages = result['text_chunks']
tables = result['tables']
figures = result.get('figures', [])
paragraphs = result.get('paragraphs', [])
sections = result.get('sections', [])
figures_meta = result.get('figures_meta', [])

print(f"pages         : {result['pages']}")
print(f"paragraphs    : {len(paragraphs)}")
print(f"sections      : {len(sections)}")
print(f"text pages    : {len(text_pages)}")
print(f"tables        : {len(tables)}")
print(f"DI figures    : {len(figures)}")
print(f"figures_meta  : {len(figures_meta)}")

print('\n--- first 5 sections (heading | level | path) ---')
for s in sections[:5]:
    heading = (s.get('heading') or '(none)')[:50]
    print(f"  {s['id']:>4}  L{s.get('level', 0)}  {heading!r}  path={s.get('path') or '(root)'}")

print('\n--- first paragraph ---')
if paragraphs:
    p = paragraphs[0]
    print(f"  id={p['id']} page={p.get('page')} role={p.get('role')!r} section_id={p.get('section_id')}")
    print(f"  bbox={p.get('bbox')}")
    print(f"  content={(p.get('content') or '')[:200]!r}")

print('\n--- first table neighbors ---')
if tables:
    t0 = tables[0]
    print(f"  id={t0.get('id')} page={t0.get('page')} caption={(t0.get('caption') or '')[:60]!r}")
    print(f"  section_path={t0.get('section_path')}")
    print(f"  neighbor_paragraph_ids={t0.get('neighbor_paragraph_ids')}")


INFO: Document Intelligence analysing https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/Mu
INFO: Request URL: 'https://azdocumentai.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30&outputContentFormat=REDACTED'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '277'
    'Accept': 'application/json'
    'x-ms-client-request-id': '724babf2-4ad5-11f1-a3f7-bcd22c162bea'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
INFO: Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-content-type-options': 'REDACTED'
    'x-ms-region': 'REDACTED'
   

pages         : 27
paragraphs    : 511
sections      : 65
text pages    : 26
tables        : 24
DI figures    : 1
figures_meta  : 1

--- first 5 sections (heading | level | path) ---
    s0  L1  'Section 0'  path=['Section 0']
    s1  L2  'Multi-Agent Research System Architecture Design & '  path=['Section 0', 'Multi-Agent Research System Architecture Design & Technical Documentation']
    s2  L3  'Table of Contents'  path=['Section 0', 'Multi-Agent Research System Architecture Design & Technical Documentation', 'Table of Contents']
    s3  L2  '1. Executive Summary'  path=['Section 0', '1. Executive Summary']
    s4  L3  'Key Highlights:'  path=['Section 0', '1. Executive Summary', 'Key Highlights:']

--- first paragraph ---
  id=p0 page=1 role=<ParagraphRole.TITLE: 'title'> section_id=s1
  bbox=[122.83919999999999, 251.6688, 486.1728, 303.4152]
  content='Multi-Agent Research System Architecture Design & Technical Documentation'

--- first table neighbors ---
  id=t0 page=1 caption='

## 3. PyMuPDF + GPT-4o vision — image extraction & description

Same call as notebook 03. Each image is uploaded to Blob and described
by GPT-4o vision; the description is what we'll embed.


In [3]:
openai = OpenAIService()
images = doc_intel.extract_images(
    pdf_bytes,
    doc_id=DOC_ID,
    blob=blob,
    openai=openai,
    figures=figures,
    figures_meta=figures_meta,
)
print(f'extracted {len(images)} images')
for img in images[:5]:
    print(
        f"- page {img['page']:>3}  source={img.get('source')}  "
        f"figure_id={img.get('figure_id')}  section_path={img.get('section_path')!r}"
    )
    print(f"    caption    : {(img.get('caption') or '')[:80]!r}")
    print(f"    neighbors  : {img.get('neighbor_paragraph_ids')}")
    print(f"    description: {(img.get('description') or '')[:80]!r}")


INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-chunk-test/figures/page15_fig0_0.png?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '188895'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '7a5897f6-4ad5-11f1-a639-bcd22c162bea'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Fri, 08 May 2026 12:00:17 GMT'
    'ETag': '"0x8DEACF95F1B5CFC"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'a1c7ecbb-801e-0094-52e2-dee294000000'

extracted 1 images
- page  15  source=figure  figure_id=f0  section_path='Section 0 > 6. Architecture Diagram'
    caption    : ''
    neighbors  : ['p262', 'p289']
    description: 'This image is a flowchart detailing a process for handling user queries through '


## 4. Section-aware chunking — `ChunkRecord`s with parent-child links

Three pure-function building blocks (no Azure I/O):

1. `chunk_paragraphs_by_section(paragraphs, sections, …)` — packs paragraphs in reading order into ~600-token windows, **never crossing section boundaries**, returning `(chunks, section_anchor_map)` where `section_anchor_map[section_id]` is the first text chunk id for that section.
2. `build_table_chunks(tables, paragraphs, …, section_anchors=…)` — bundles `[Table on page N — caption] / Section / Context (neighbors) / Table markdown` and stamps `parent_id = section_anchor`.
3. `build_image_chunks(images, paragraphs, …, section_anchors=…)` — bundles `[Figure …] / Section / Caption / Surrounding text / Description` and stamps `parent_id`.

`assemble_chunks` is the one-call wrapper — when `paragraphs` + `sections` are supplied it takes the section-aware path; otherwise it falls back to the legacy per-page sliding window.


In [4]:
from collections import Counter

# Step 1: section-aware text packing — gives us the anchor map.
text_chunks_only, section_anchors = chunk_paragraphs_by_section(
    paragraphs,
    sections,
    doc_id=DOC_ID,
    doc_filename=DOC_FILENAME,
    doc_hash=DOC_HASH,
)

# Step 2: tables + images bundled with neighbors and linked via parent_id.
table_chunks_only = build_table_chunks(
    tables,
    paragraphs,
    doc_id=DOC_ID,
    doc_filename=DOC_FILENAME,
    doc_hash=DOC_HASH,
    section_anchors=section_anchors,
)
image_chunks = build_image_chunks(
    images,
    paragraphs,
    doc_id=DOC_ID,
    doc_filename=DOC_FILENAME,
    doc_hash=DOC_HASH,
    section_anchors=section_anchors,
)

# Step 3: one-call assembly (also backfills doc metadata onto image_chunks).
all_chunks = assemble_chunks(
    doc_id=DOC_ID,
    text_pages=text_pages,
    tables=tables,
    image_chunks=image_chunks,
    paragraphs=paragraphs,
    sections=sections,
    doc_filename=DOC_FILENAME,
    doc_hash=DOC_HASH,
)

counts = Counter(c.type for c in all_chunks)
print(f'sections with anchor: {len(section_anchors)}')
print(f'total chunks        : {len(all_chunks)}')
for t in ('text', 'table', 'image'):
    print(f'  {t:5s}             : {counts.get(t, 0)}')

# Show an example parent → children link.
if table_chunks_only and table_chunks_only[0].parent_id:
    parent_id = table_chunks_only[0].parent_id
    parent = next((c for c in all_chunks if c.id == parent_id), None)
    print('\n--- example parent / child ---')
    print(f"table chunk page {table_chunks_only[0].page} section_path={table_chunks_only[0].section_path!r}")
    print(f"  parent_id = {parent_id}")
    if parent:
        print(f"  parent is text chunk page {parent.page}, {len(parent.content)} chars, section_id={parent.section_id}")


sections with anchor: 65
total chunks        : 93
  text              : 68
  table             : 24
  image             : 1

--- example parent / child ---
table chunk page 1 section_path='Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation'
  parent_id = 74c6b8a5-857b-49f7-848a-06bdfb8f6f86


### Peek at one chunk of each type


In [14]:
for t in ('text', 'table', 'image'):
    sample = next((c for c in all_chunks if c.type == t), None)
    if not sample:
        continue
    print(f"--- {t.upper()} (page {sample.page}, {len(sample.content)} chars) ---")
    print(f"  id           : {sample.id}")
    print(f"  doc_filename : {sample.doc_filename}")
    print(f"  doc_hash     : {(sample.doc_hash or '')[:16]}…")
    print(f"  section_id   : {sample.section_id}")
    print(f"  section_path : {sample.section_path!r}")
    print(f"  section_level: {sample.section_level}")
    print(f"  parent_id    : {sample.parent_id}")
    print(f"  element_id   : {sample.element_id}")
    print(f"  reading_order: {sample.reading_order}")
    print(f"  bbox         : {sample.bbox}")
    if sample.image_url:
        print(f"  image_url    : {sample.image_url[:80]}…")
    print(f"  content[:400]:\n{sample.content[:400]}")
    print()


--- TEXT (page 1, 170 chars) ---
  id           : c56bdad8-ec1f-49e6-bd09-21009ab8a8ec
  doc_filename : Multi_Agent_Research_System_Architecture.pdf
  doc_hash     : 0b4ae90fa7bd2cff…
  section_id   : s1
  section_path : 'Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation'
  section_level: 2
  parent_id    : None
  element_id   : None
  reading_order: 0
  bbox         : [122.83919999999999, 251.6688, 486.1728, 303.4152]
  content[:400]:
[Section: Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation]
Multi-Agent Research System Architecture Design & Technical Documentation

--- TABLE (page 1, 391 chars) ---
  id           : f8f826d9-1356-4b38-a1d0-ae209e9c0f80
  doc_filename : Multi_Agent_Research_System_Architecture.pdf
  doc_hash     : 0b4ae90fa7bd2cff…
  section_id   : s1
  section_path : 'Section 0 > Multi-Agent Research System Architecture Design & Technical Documentation'
  section_level: None
  parent_id    : c56

## 5. Sanity checks

- text chunk char length stays within the budget (`CHUNK_TOKENS * 4 * HARD_SPLIT_HEADROOM`)
- every chunk carries `doc_id` + `doc_filename` + `doc_hash`
- every table / image chunk that has a `section_id` also has a `parent_id` from the anchor map
- every image chunk carries `image_url`


In [15]:
import statistics

# Hard cap = CHUNK_TOKENS * 4 chars * 1.2 headroom (see HARD_SPLIT_HEADROOM in chunking.py).
MAX_CHARS_HARD = int(CHUNK_TOKENS * 4 * 1.2)

text_chunks = [c for c in all_chunks if c.type == 'text']
table_chunks = [c for c in all_chunks if c.type == 'table']
img_chunks = [c for c in all_chunks if c.type == 'image']

# 1. Size budget.
oversize = [c for c in text_chunks if len(c.content) > MAX_CHARS_HARD]
lengths = [len(c.content) for c in text_chunks]
if lengths:
    print(
        f'text chunks: count={len(text_chunks)} '
        f'min={min(lengths)} max={max(lengths)} '
        f'mean={statistics.mean(lengths):.0f} median={statistics.median(lengths):.0f} '
        f'(hard cap {MAX_CHARS_HARD})'
    )
print(f'oversize text chunks       : {len(oversize)} (expect 0)')

# 2. Multi-doc identity on every chunk.
missing_doc_meta = [
    c for c in all_chunks
    if not c.doc_id or not c.doc_filename or not c.doc_hash
]
print(f'chunks missing doc_id/name/hash: {len(missing_doc_meta)} (expect 0)')

# 3. Parent-child wiring on tables and images that live inside a section.
table_orphans = [
    c for c in table_chunks
    if c.section_id and c.section_id in section_anchors and not c.parent_id
]
img_orphans = [
    c for c in img_chunks
    if c.section_id and c.section_id in section_anchors and not c.parent_id
]
print(f'table chunks missing parent_id : {len(table_orphans)} (expect 0)')
print(f'image chunks missing parent_id : {len(img_orphans)} (expect 0)')

# 4. Image chunks always have a blob URL.
missing_url = [c for c in img_chunks if not c.image_url]
print(f'image chunks missing url       : {len(missing_url)} (expect 0)')
if img_chunks:
    print('sample image header :', img_chunks[0].content.split(']')[0] + ']')

# 5. Section coverage — text chunks per section count.
per_section = Counter(c.section_id for c in text_chunks if c.section_id)
print(f'\nsections with at least one text chunk: {len(per_section)} / {len(sections)}')


text chunks: count=68 min=72 max=2329 mean=590 median=355 (hard cap 2880)
oversize text chunks       : 0 (expect 0)
chunks missing doc_id/name/hash: 0 (expect 0)
table chunks missing parent_id : 0 (expect 0)
image chunks missing parent_id : 0 (expect 0)
image chunks missing url       : 0 (expect 0)
sample image header : [Figure on page 15]

sections with at least one text chunk: 64 / 65


## 6. Embedding — `text-embedding-ada-002`

Mirrors the embed step inside `IngestionPipeline.process_pdf`: batches
of 16, attached to each `ChunkRecord.embedding`.


In [ ]:
BATCH = 16
texts = [c.content for c in all_chunks]
vectors: list[list[float]] = []
for i in range(0, len(texts), BATCH):
    vectors.extend(openai.embed(texts[i:i + BATCH]))

for c, v in zip(all_chunks, vectors):
    c.embedding = v

dims = {len(v) for v in vectors}
print(f'embedded {len(vectors)} chunks; embedding dim(s): {dims}')
assert len(vectors) == len(all_chunks)
assert all(c.embedding for c in all_chunks)
print('OK every chunk has an embedding')


## 7. Quick semantic-search sanity (optional)

Cosine-similarity over the in-memory chunks — a fast way to eyeball
whether chunking + embedding produced sensible vectors before pushing
to AI Search.


In [ ]:
import math


def cosine(a, b):
    s = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return s / (na * nb) if na and nb else 0.0


QUERY = 'multi-agent system architecture'
qvec = openai.embed([QUERY])[0]

scored = sorted(
    ((cosine(qvec, c.embedding), c) for c in all_chunks),
    key=lambda x: x[0],
    reverse=True,
)[:5]

for score, c in scored:
    section = (c.section_path or '(no section)')[:600]
    print(f'{score:.3f}  [{c.type:5s} p{c.page:>3}]  section={section!r}')
    print(f'         {c.content[:1200].strip()}…')
    print('\n')
